# Erdős squarefree problem

In [ ]:
#@title Verification code

# pylint: disable=unused-import
# pylint: disable=g-bad-import-order




@njit
def gcd_numba(a, b):
  """Computes GCD using Numba.

  Handles non-positive inputs returning abs(gcd).

  Args:
    a: The first integer.
    b: The second integer.

  Returns:
    The greatest common divisor of a and b.
  """
  # Ensure inputs are integers for modulo operation
  a_int = int(a)
  b_int = int(b)
  while b_int:
    a_int, b_int = b_int, a_int % b_int
  # Ensure result is non-negative, consistent with math.gcd
  # Note: gcd(0, 0) typically returns 0. gcd(a, 0) returns abs(a).
  return abs(a_int)


@njit
def is_squarefree_numba(n_signed):
  """Checks if n is squarefree using Numba. Handles n<=0 and n=1."""
  # Definition: An integer is squarefree if no prime squared divides it.
  # Conventionally, 1 is squarefree. 0 is not (divisible by 4, 9, ...).
  # Negative numbers: -n is squarefree iff n is. e.g., -4 is not, -6 is.
  n = abs(int(n_signed))  # Ensure positive integer
  if n == 0:
    return False
  if n == 1:
    return True

  temp_n = n  # Work on a copy

  # Check factor 2 first
  if temp_n % 2 == 0:
    temp_n //= 2
    if temp_n % 2 == 0:  # Divisible by 4
      return False

  # Check odd factors starting from 3
  d = 3
  # Optimization: Only need to check primes d up to sqrt(n)
  # The loop condition d * d <= temp_n handles this.
  while d * d <= temp_n:
    if temp_n % d == 0:  # Found a factor d
      temp_n //= d  # Divide by d
      # Check if divisible by d again (meaning d*d was a factor of original n)
      if temp_n % d == 0:
        return False
    # Move to the next odd number
    d += 2

  # After the loop, temp_n might be a prime > sqrt(original n) or 1.
  # No square factor involving primes up to sqrt(n) was found.
  return True




def calculate_score(set_a: List[int], modulus: int) -> float:
  """Calculates the score for a set A modulo M based on squarefree condition."""
  # --- Basic Validity Checks ---
  if not isinstance(modulus, int) or modulus <= 1:
    return -1_000_000.0
  if not isinstance(set_a, list):
    return -1_000_000.0

  if modulus % 25 == 0:
    modulus = 625

  # Handle empty set A
  if not set_a:
    return 0.0  # Density is 0, penalty is 0
  processed_set_a = sorted(
      list(
          set(
              int(x)
              for x in set_a
              if isinstance(x, (int, np.integer)) and 0 <= int(x) < modulus
          )
      )
  )

  n = len(processed_set_a)
  density = n / modulus

  penalty_count = 0
  # Iterate through all pairs (a, b) using the input list A
  for i in range(n):
    a = processed_set_a[i]
    for j in range(i, n):
      b = processed_set_a[j]
      # Calculate ab + 1. Python handles large integers.
      val = int(a) * int(b) + 1
      # Calculate gcd(ab+1, M). Use numba helper.
      g = gcd_numba(val, int(modulus))

      # Check if gcd is squarefree. Use numba helper.
      # g can be 0 only if val=0 and M=0 (disallowed). g=1 is squarefree.
      if is_squarefree_numba(g):
        penalty_count += 1

  # Penalty term as defined: (count of bad pairs) / M
  penalty = penalty_count / modulus
  score = density - penalty

  # Return score as float
  return float(score)


def format_feedback_repr(feedback: Mapping[str, Any]) -> dict[str, str]:
  """Formats feedback dictionary for representation in code."""
  formatted_feedback = {}
  # Ensure numpy arrays are printed fully if they appear in feedback
  np.set_printoptions(threshold=np.inf)
  for key, value in feedback.items():
    if isinstance(value, np.ndarray):
      # Use numpy's array representation
      repr_str = repr(value)
      # Clean up excessive whitespace and newlines for storage
      cleaned_repr_str = re.sub(r'\s+', ' ', repr_str).replace('\n', '')
      # Simple reconstruction string - might need adjustment for complex types
      array_content = cleaned_repr_str[
          cleaned_repr_str.find('(') + 1 : cleaned_repr_str.rfind(')')
      ]
      dtype_str = (
          f', dtype={value.dtype.name}'
          if value.dtype.name not in ['int64', 'float64']
          else ''
      )  # Specify dtype if not default
      # Reconstruct np.array call string
      formatted_feedback[key] = f'np.array({array_content}{dtype_str})'

    elif isinstance(value, list):
      # Use standard repr for lists. Assumes list contains simple types (int,
      # float, etc.)
      formatted_feedback[key] = repr(value)
    elif isinstance(value, (int, float, bool, str)):
      # Use standard repr for basic Python types.
      formatted_feedback[key] = repr(value)
    # Add handling for numpy scalar types if they appear
    elif isinstance(value, (np.integer, np.floating, np.bool_)):
      formatted_feedback[key] = repr(
          value.item()
      )  # Convert numpy scalar to Python scalar
    else:
      # Fallback: Use standard repr for other types.
      # May not be reconstructable.
      formatted_feedback[key] = repr(value)
  return formatted_feedback


def evaluate(params: Any) -> tuple[dict[str, float], dict[str, str]]:
  """Evaluates a modulus M to find the best set A maximizing the score."""
  result = {}
  feedback = {}
  del params

  modulus, best_a = search_for_best_set()

  processed_a = sorted(
      list(
          set(
              int(x)
              for x in best_a
              if isinstance(x, (int, np.integer)) and 0 <= int(x) < modulus
          )
      )
  )
  best_a_list = processed_a
  score = calculate_score(best_a_list, modulus)

  result['score'] = score
  feedback['best_score_found'] = score
  feedback['best_modulus'] = modulus
  feedback['best_set'] = best_a_list

  print(f'Evaluation complete for M = {modulus}. Final Score: {score:.6f}')

  # Format feedback dictionary into strings for code representation
  feedback_formatted = format_feedback_repr(feedback)
  return result, feedback_formatted

In [ ]:
#@title Initial program

"""FunSearch experiment codebase for finding sets A mod M where ab+1 is rarely squarefree."""
import itertools
import logging
import time
from scipy import integrate
import numpy as np
from scipy import optimize
import warnings
import random
import re
from typing import Any, Callable, Mapping, List, Tuple
import scipy.linalg as la
import collections
import copy
import math
import numba
import ast  # For safely evaluating string representations from previous runs

njit = numba.njit
def search_for_best_set() -> Tuple[int, List[int]]:
  """Searches for the best set A modulo M using random search."""
  variable_name = 'best_set_iqhd'
  variable_name_modulus = 'best_modulus_iqhd'
  current_a = []  # Initialize A
  modulus = 121

  if variable_name in globals():
    current_a = globals()[variable_name]
  elif variable_name_modulus in globals():
    modulus = globals()[variable_name_modulus]
  best_modulus = modulus

  # --- Fallback / Initial Random Initialization ---
  # Initialize if loading failed, variable not found, or loaded set was
  # empty/invalid
  if not current_a:
    initial_size = max(1, modulus // 5)
    # Ensure k is not larger than population size for random.sample
    k = min(initial_size, modulus)
    if k > 0:
      current_a = random.sample(range(modulus), k=k)
    else:  # Handles M=0 or M=1 (though M>1 is expected)
      current_a = []
    current_a = sorted([int(x) for x in current_a])  # Ensure ints and sort

  if not current_a and modulus > 0:
    current_a = [random.randint(0, modulus - 1)]

  best_a = current_a.copy()
  best_score = calculate_score(best_a, modulus)
  print(
      f'Initial score for M={modulus}: {best_score:.6f}, Initial A size:'
      f' {len(best_a)}, A: {best_a[:20]}...'
  )  # Print initial A truncated

  # --- Random Search Loop ---
  start_time = time.time()
  eval_count = 0
  max_search_time = np.random.uniform(100, 1000)
  print(f'Starting search for {max_search_time:.0f} seconds.')
  improvements = 0

  while time.time() - start_time < max_search_time:
    mutated_a = current_a.copy()
    mutation_type = random.random()  # Random float [0.0, 1.0)
    made_change = False

    # Determine elements not currently in A for efficient addition/change
    # This can be costly if M is large, do it less often or approximate?
    # For simplicity, let's assume M is manageable for now.
    elements_in_a = set(mutated_a)
    potential_additions = [x for x in range(modulus) if x not in elements_in_a]

    # Mutation Strategy: Add, Remove, or Change element
    if mutation_type < 0.4 and len(mutated_a) < modulus and potential_additions:
      # Add an element not currently in A
      new_element = random.choice(potential_additions)
      mutated_a.append(new_element)
      made_change = True
    elif mutation_type < 0.8 and mutated_a:
      # Remove a random element from A
      remove_idx = random.randrange(
          len(mutated_a)
      )  # Safer than randint for empty list edge case
      mutated_a.pop(remove_idx)
      made_change = True
    elif mutated_a and potential_additions:
      # Change an element: Replace a random element in A with one not in A
      remove_idx = random.randrange(len(mutated_a))
      add_element = random.choice(potential_additions)
      mutated_a[remove_idx] = add_element  # Replace element
      made_change = True

    if mutation_type < 0.05:
      modulus = random.randint(10, 5000)
      mutated_a = [random.randint(0, modulus - 1)]
    # else: No change occurred (e.g., tried to add to full set,
    # remove from empty, or change failed)

    # If a valid mutation occurred
    if made_change:
      # Ensure A is sorted list of unique ints after mutation
      current_a = sorted(list(set(int(x) for x in mutated_a)))

      # Safeguard against empty list if modulus > 0
      if not current_a and modulus > 0:
        current_a = [random.randint(0, modulus - 1)]

      # Evaluate the mutated set
      score = calculate_score(current_a, modulus)
      # print(score, modulus, current_a)
      eval_count += 1

      # Update best if improved
      if score > best_score:
        improvements += 1
        best_score = score
        best_a = current_a.copy()
        best_modulus = modulus
        # Print significant improvements or periodically
        print(
            f'Eval {eval_count}, M={modulus}, New best score: {best_score:.6f},'
            f' Size: {len(best_a)}, A: {best_a[:20]}...'
        )

      # Simple random walk: The mutated set becomes the current set for the
      # next iteration. Could implement hill-climbing (only accept if score
      # improves) or simulated annealing here.

  # --- Search Completion ---
  print(
      f'Search finished. Final score: {best_score:.6f}, Final A size:'
      f' {len(best_a)}, A: {best_a}'
  )
  print(f'Total evaluations: {eval_count}, Improvements found: {improvements}')
  # Return the best A found during the search
  return best_modulus, best_a

**Prompt used**

Problem: Erdős Squarefree Problem Variant

Act as an expert in number theory and computational search algorithms. Your task is to find a set of non-negative integers $A$ modulo $M$ for a given integer modulus $M > 1$. The goal is to maximize a score based on the density of the set $A$ and a property related to squarefree numbers.

Specifically, for a given modulus $M$, you need to find a subset $A \subseteq (0, 1, \dots, M-1)$ that maximizes the following score function:

Score = |A|/M - (number of pairs (a, b) \in A \times A such that \gcd(ab+1, M) is squarefree) / M$

Here, $A \times A$ includes pairs where $a=b$. A number $g$ is squarefree if it is not divisible by any perfect square greater than 1 (i.e., no prime $p$ satisfies $p^2 | g$). The $\gcd(x, y)$ is the greatest common divisor of $x$ and $y$.

The problem is inspired by an Erdos question about sets $A \subseteq (1, \dots, N)$ where $ab+1$ is never squarefree for $a, b \in A$. This FunSearch task explores this property using modular arithmetic as a proxy. The conjecture related to the original problem suggests that residues congruent to 7 modulo 25 might be optimal, but our task is to find an even better construction.

You need to implement a search function search_for_best_set(modulus) that takes an integer modulus (M) as input and returns the best set A (as a list of integers) it finds for that modulus within a time limit of 1000 seconds.

The evaluation function calculate_score(A, M) is provided (you don't need to implement it, but you can call it):

def calculate_score(A: List[int], M: int) -> float:
    """Calculates the score for a set A modulo M based on squarefree condition."""
    # ... (implementation as provided in the full code) ...
    # Handles checks, calculates density, counts pairs where gcd(ab+1, M) is squarefree,
    # and returns score = density - (penalty_count / M).
    if not isinstance(M, int) or M <= 1: return -1_000_000.0
    if not isinstance(A, list): return -1_000_000.0
    if not A: return 0.0
    n = len(A)
    density = n / M
    penalty_count = 0
    for i in range(n):
        a = A[i]
        for j in range(n):
            b = A[j]
            val = int(a) * int(b) + 1
            g = gcd_numba(val, int(M))
            if is_squarefree_numba(g):
                penalty_count += 1
    penalty = penalty_count / M
    score = density - penalty
    return float(score)

## What AlphaEvolve found

AlphaEvolve easily found the known best construction: the intersection of $\{1, \ldots, N\}$ with the residue class $7 \bmod 25$, which gives $C(N) \geq \lceil (N-7)/25 \rceil$. It did not manage to find a better construction. Shortly before the paper was finalized, it was independently demonstrated by Sawhney that this lower bound is sharp for all sufficiently large $N$.